In [40]:
import os
os.environ.setdefault('RUN_ALFWORLD_EVAL', '1')

# 可选：如果数据集太大，为了快速测试，可以限制评估的游戏数量，例如只跑5个
os.environ.setdefault('ALFWORLD_MAX_GAMES', '5')

'5'

In [ ]:
import os
import re

try:
    from openai import OpenAI
except ImportError as exc:
    OpenAI = None
    _openai_import_error = exc
else:
    _openai_import_error = None

OPENAI_API_KEY = os.environ.get("DEEPSEEK_API_KEY") or os.environ.get("OPENAI_API_KEY")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "deepseek-chat")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.deepseek.com")
_client_kwargs = {"api_key": OPENAI_API_KEY}
if OPENAI_BASE_URL:
    _client_kwargs["base_url"] = OPENAI_BASE_URL
client = OpenAI(**_client_kwargs) if OPENAI_API_KEY and OpenAI is not None else None

ACTION_PREFIX_RE = re.compile(r'^(?:>|(?:Act(?:ion)?\s*\d*\s*:))\s*', re.IGNORECASE)
ACTION_STARTERS = (
    'think:', 'go to ', 'open ', 'close ', 'take ', 'put ',
    'move ', 'clean ', 'heat ', 'cool ', 'examine ', 'look ', 'inventory'
)

def _normalize_action_line(line):
    line = ACTION_PREFIX_RE.sub('', line.strip()).strip().strip('`').strip()
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1].strip()
    if line.startswith("'") and line.endswith("'"):
        line = line[1:-1].strip()
    return line.rstrip('.')

def _looks_like_action(line):
    return line.lower().startswith(ACTION_STARTERS)

def _match_admissible(candidate, admissible_by_lower):
    variants = [candidate]
    variants.append(candidate.replace(' on ', ' in/on '))
    variants.append(candidate.replace(' in ', ' in/on '))
    put_match = re.match(r'^put (.+?) (?:in/on|on|in) (.+)$', candidate, re.IGNORECASE)
    if put_match:
        variants.append(f"move {put_match.group(1)} to {put_match.group(2)}")
    examine_match = re.match(r'^examine (.+)$', candidate, re.IGNORECASE)
    if examine_match:
        variants.append(f"go to {examine_match.group(1)}")
    for variant in variants:
        match = admissible_by_lower.get(variant.lower())
        if match:
            return match
    return None

def _extract_action(response_text, admissible_commands=None):
    admissible_commands = list(admissible_commands or [])
    admissible_by_lower = {command.lower(): command for command in admissible_commands}
    candidates = []
    for raw_line in response_text.replace('\r', '\n').split('\n'):
        candidate = _normalize_action_line(raw_line)
        if candidate:
            candidates.append(candidate)

    for candidate in candidates:
        if candidate.lower().startswith('think:'):
            return candidate
        match = _match_admissible(candidate, admissible_by_lower)
        if match:
            return match
        if not admissible_commands and _looks_like_action(candidate):
            return candidate

    if admissible_commands:
        normalized_response = ' '.join(response_text.lower().split())
        for command in sorted(admissible_commands, key=len, reverse=True):
            if command.lower() in normalized_response:
                return command

    if admissible_commands:
        return ''
    return candidates[0] if candidates else ''

def _build_chat_messages(prompt, admissible_commands=None, correction=None):
    instruction = (
        "You are playing ALFWorld. Continue the transcript by outputting exactly one next action after the final >. "
        "Do not explain, do not add markdown, and do not output blank lines. "
        "Valid action forms include: think: ..., go to ..., open ..., close ..., take ..., move ..., clean ..., heat ..., cool ..., examine ..., inventory."
    )
    if admissible_commands:
        available_actions = '\n'.join(f'- {command}' for command in admissible_commands)
        instruction += (
            "\nIf you choose an environment action, it must exactly match one of these available actions:\n"
            f"{available_actions}"
        )
    user_content = prompt
    if correction:
        user_content += f"\n\n{correction}"
    return [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]

def llm(prompt, stop=["\n"], admissible_commands=None):
    if client is None:
        if _openai_import_error is not None:
            raise RuntimeError("Install the openai package in this notebook kernel to call llm().") from _openai_import_error
        raise RuntimeError("Set OPENAI_API_KEY before calling llm().")

    last_raw_text = ''
    is_completion_model = OPENAI_MODEL.startswith(("text-", "davinci", "babbage", "curie", "ada")) or OPENAI_MODEL.endswith("-instruct")
    for attempt in range(3):
        correction = None
        if attempt:
            correction = f"Your previous response was invalid: {last_raw_text!r}. Return exactly one valid action from the available actions."

        if is_completion_model:
            completion_prompt = prompt if correction is None else f"{prompt}\n\n{correction}\n>"
            response = client.completions.create(
                model=OPENAI_MODEL,
                prompt=completion_prompt,
                temperature=0,
                max_tokens=100,
                top_p=1,
                frequency_penalty=0.0,
                presence_penalty=0.0,
                stop=stop,
            )
            raw_text = response.choices[0].text
        else:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=_build_chat_messages(prompt, admissible_commands, correction),
                temperature=0,
                max_tokens=80,
                top_p=1,
                frequency_penalty=0.0,
                presence_penalty=0.0,
            )
            message = response.choices[0].message
            raw_text = message.content or getattr(message, 'reasoning_content', '') or ""

        action = _extract_action(raw_text, admissible_commands)
        if action:
            return action
        last_raw_text = raw_text

    if admissible_commands:
        if 'look' in admissible_commands:
            return 'look'
        return admissible_commands[0]
    raise RuntimeError(f"LLM did not return a usable action. Last raw response: {last_raw_text!r}")

In [42]:
import os
from pathlib import Path
import yaml

local_alfworld_data = Path('alfworld_data').resolve()
configured_alfworld_data = os.environ.get('ALFWORLD_DATA')
configured_data_path = Path(configured_alfworld_data).expanduser() if configured_alfworld_data else None
if local_alfworld_data.exists() and (configured_data_path is None or not (configured_data_path / 'json_2.1.1').exists()):
    os.environ['ALFWORLD_DATA'] = str(local_alfworld_data)

try:
    from alfworld.agents.environment import get_environment
except ImportError as exc:
    get_environment = None
    _alfworld_import_error = exc
else:
    _alfworld_import_error = None

with open('base_config.yaml') as reader:
    config = yaml.safe_load(reader)
    
split = "eval_out_of_distribution"
env = None
env_error = None
env_game_files = []

alfworld_data = os.environ.get('ALFWORLD_DATA')
if get_environment is None:
    env_error = _alfworld_import_error
elif not alfworld_data or not Path(alfworld_data).expanduser().exists():
    env_error = RuntimeError("Set ALFWORLD_DATA to the ALFWorld data directory, or place it at ./alfworld_data.")
else:
    try:
        alfworld_env = get_environment(config["env"]["type"])(config, train_eval=split)
        env_game_files = list(getattr(alfworld_env, 'game_files', []))
        env = alfworld_env.init_env(batch_size=1)
    except Exception as exc:
        env = None
        env_error = exc

if env is None:
    print(f"ALFWorld environment unavailable: {env_error}")

def process_ob(ob):
    if ob.startswith('You arrive at loc '):
        ob = ob[ob.find('. ')+2:]    
    return ob

Initializing AlfredTWEnv...


100%|██████████| 341/341 [00:00<00:00, 2528.40it/s]

Overall we have 134 games in split=eval_out_of_distribution
Evaluating with 134 games


In [43]:
import json
folder = './prompts/'
prompt_file = 'alfworld_3prompts.json'
with open(folder + prompt_file, 'r') as f:
    d = json.load(f)

In [44]:
import sys

def choose_unvisited_navigation(admissible_commands, visited_locations):
    for command in admissible_commands:
        if command.startswith('go to ') and command not in visited_locations:
            return command
    return None

def alfworld_run(prompt, to_print=True, ob='', admissible_commands=None):
    if env is None:
        raise RuntimeError(f"ALFWorld environment is unavailable: {env_error}")
    init_prompt = prompt + ob + '\n>'
    prompt = ''
    current_admissible_commands = list(admissible_commands or [])
    visited_locations = set()
    consecutive_thinks = 0
    if to_print:
        print(ob)
        sys.stdout.flush()
    for i in range(1, 50):
        action = llm(init_prompt + prompt, stop=['\n'], admissible_commands=current_admissible_commands).strip()
        if action.startswith('think:'):
            consecutive_thinks += 1
            lower_action = action.lower()
            should_explore = consecutive_thinks >= 3 or any(phrase in lower_action for phrase in ("not found", "haven't found", "can't find", "searched all"))
            if should_explore:
                exploration_action = choose_unvisited_navigation(current_admissible_commands, visited_locations)
                if exploration_action:
                    action = exploration_action
                    consecutive_thinks = 0
        else:
            consecutive_thinks = 0
        if action.startswith('think:'):
            observation = 'OK.'
            reward, done = 0, False
        else:
            observation, reward, done, info = env.step([action])
            observation, reward, done = process_ob(observation[0]), info['won'][0], done[0]
            current_admissible_commands = info.get('admissible_commands', [current_admissible_commands])[0]
            if action.startswith('go to '):
                visited_locations.add(action)
        if to_print:
            print(f'Act {i}: {action}\nObs {i}: {observation}')
            sys.stdout.flush()
        prompt += f' {action}\n{observation}\n>'
        if done:
            return reward
    return 0

In [45]:
prefixes = {
    'pick_and_place': 'put',
    'pick_clean_then_place': 'clean',
    'pick_heat_then_place': 'heat',
    'pick_cool_then_place': 'cool',
    'look_at_obj': 'examine',
    'pick_two_obj': 'puttwo'
}
cnts = [0] * 6
rs = [0] * 6

run_eval = os.environ.get('RUN_ALFWORLD_EVAL') == '1'
game_files = env_game_files if env is not None else []
available_games = len(game_files)
requested_games = int(os.environ.get('ALFWORLD_MAX_GAMES', available_games))
num_games = min(requested_games, available_games)

if env is None:
    print('ALFWorld environment is unavailable; install alfworld and set ALFWORLD_DATA to run evaluation.')
elif not game_files:
    print(f'ALFWorld environment loaded, but no games were found for split={split}. Check base_config.yaml and ALFWORLD_DATA.')
elif client is None:
    print('OPENAI_API_KEY is not set; environment and prompts loaded. Set OPENAI_API_KEY and rerun this cell to evaluate.')
elif not run_eval:
    print(f'Environment, prompts, and OpenAI client loaded with {available_games} games. Set RUN_ALFWORLD_EVAL=1 to run evaluation.')
elif num_games <= 0:
    print('ALFWORLD_MAX_GAMES is 0; no evaluation episodes requested.')
else:
    for episode_idx in range(num_games):
        ob, info = env.reset()
        admissible_commands = info.get('admissible_commands', [[]])[0]
        ob = '\n'.join(ob[0].split('\n\n')[1:])
        name = '/'.join(info['extra.gamefile'][0].split('/')[-3:-1])
        print(name)
        r = 0
        matched = False
        for i, (k, v) in enumerate(prefixes.items()):
            if name.startswith(k):
                prompt = 'Interact with a household to solve a task. Here are two examples.\n' + d[f'react_{v}_1'] + d[f'react_{v}_0'] + '\nHere is the task.\n'
                print(k, v)
                try:
                    r = alfworld_run(prompt, ob=ob, admissible_commands=admissible_commands)
                except Exception as exc:
                    print(f'Episode failed with error: {exc}')
                    r = 0
                rs[i] += r
                cnts[i] += 1
                matched = True
                break
        total = sum(cnts)
        score = sum(rs) / total if total else 0
        print(episode_idx + 1, 'r', r, 'rs', rs, 'cnts', cnts, 'sum(rs)/sum(cnts)', score)
        if not matched:
            print('No prompt prefix matched this game; skipped evaluation for it.')
        print('------------\n')
    total = sum(cnts)
    final_score = sum(rs) / total if total else 0
    print(f'Final result: {sum(rs)}/{total} = {final_score:.3f}')


pick_and_place_simple-Mug-None-Desk-308/trial_T20190908_125200_737896
pick_and_place put
You are in the middle of a room. Looking quickly around you, you see a bed 1, a desk 2, a desk 1, a drawer 6, a drawer 5, a drawer 4, a drawer 3, a drawer 2, a drawer 1, a garbagecan 1, a laundryhamper 1, a safe 1, a shelf 6, a shelf 5, a shelf 4, a shelf 3, a shelf 2, and a shelf 1.
Your task is to: put a mug in desk.
Act 1: look
Obs 1: You are in the middle of a room. Looking quickly around you, you see nothing.
Act 2: look
Obs 2: You are in the middle of a room. Looking quickly around you, you see nothing.
Act 3: go to desk 1
Obs 3: You arrive at desk 1. On the desk 1, you see a alarmclock 1, a bowl 1, a cd 1, a desklamp 1, and a laptop 1.
Act 4: go to desk 2
Obs 4: You arrive at desk 2. On the desk 2, you see a bowl 2, a cellphone 2, and a pen 2.
Act 5: go to drawer 1
Obs 5: You arrive at drawer 1. The drawer 1 is closed.
Act 6: open drawer 1
Obs 6: You open the drawer 1. The drawer 1 is open. 

KeyboardInterrupt: 